In [0]:
# 1. Widgets for Catalog and Schema parameterization using comboboxes
dbutils.widgets.combobox(
    "catalog", "dbr_dev", ["workspace", "dbr_dev"], "Unity Catalog"
)
dbutils.widgets.combobox(
    "bronze_schema",
    "joshuandegwa_bronze",
    ["joshuandegwa_bronze", "bronze"],
    "Bronze schema",
)
dbutils.widgets.combobox(
    "silver_schema",
    "joshuandegwa_silver",
    ["joshuandegwa_silver", "silver"],
    "Silver schema",
)

# 2. Get parameter values from widgets
catalog_name = dbutils.widgets.get("catalog")
bronze_schema_name = dbutils.widgets.get("bronze_schema")
silver_schema_name = dbutils.widgets.get("silver_schema")

# 3. Ensure Silver schema exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{silver_schema_name}")

# 4. Define source and target paths
bronze_table_path = f"{catalog_name}.{bronze_schema_name}.lab4_flights_raw"
silver_table_path = f"{catalog_name}.{silver_schema_name}.lab4_flights_clean"

print(f"Reading from: {bronze_table_path}")
print(f"Writing to: {silver_table_path}")

In [0]:
# Silver cleaning 1

from pyspark.sql import functions as F

print(f"Reading raw data from: {bronze_table_path}")
df_raw = spark.table(bronze_table_path)

# 4. Apply Silver Cleaning, Explicit Schema & Data Quality Rules
df_cleaned = (
    df_raw
    # Explicitly enforce types for a clearly defined schema
    .withColumn("icao24", F.col("icao24").cast("string"))
    .withColumn("callsign", F.col("callsign").cast("string"))
    .withColumn("origin_country", F.col("origin_country").cast("string"))
    .withColumn("longitude", F.col("longitude").cast("double"))
    .withColumn("latitude", F.col("latitude").cast("double"))
    .withColumn("baro_altitude", F.col("baro_altitude").cast("double"))
    .withColumn("velocity", F.col("velocity").cast("double"))
    .withColumn("true_track", F.col("true_track").cast("double"))
    .withColumn("vertical_rate", F.col("vertical_rate").cast("double"))
    .withColumn("on_ground", F.col("on_ground").cast("boolean"))
    # Rule 1: Drop rows where mandatory business key 'icao24' is null
    .filter(F.col("icao24").isNotNull())
    # Rule 2: Filter out geographic coordinate anomalies (Latitude between -90 and 90)
    .filter((F.col("latitude") >= -90.0) & (F.col("latitude") <= 90.0))
    # Rule 3: Filter out altitude anomalies (Barometric altitude >= 0 and <= 20000)
    .filter((F.col("baro_altitude") >= 0.0) & (F.col("baro_altitude") <= 20000.0))
    # Rule 4: Remove exact duplicate rows
    .dropDuplicates()
    # Rule 5: Add robust audit metadata columns for lineage and tracking
    .withColumn("silver_processed_timestamp", F.current_timestamp())
    .withColumn("data_layer", F.lit("silver"))
    .withColumn("source_table", F.lit("lab4_flights_raw"))
)

# 5. Write the cleaned dataset to the Silver Delta table with overwriteSchema enabled
(
    df_cleaned.write.format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema", "true"
    )  # 👈 This forces Delta to update the table structure/metadata cleanly
    .saveAsTable(silver_table_path)
)

print(
    f"✅ Silver cleaning complete! Cleaned data successfully written to:"
    f" {silver_table_path}"
)

# 6. Quick validation preview
display(spark.table(silver_table_path).limit(10))

In [0]:
# Merge insert option1

# 1. Ensure the source DataFrame has unique icao24 keys to prevent multiple matches
# We drop duplicates based on the primary key, keeping the first occurrence
df_unique_source = df_cleaned.dropDuplicates(["icao24"])

# 2. Register the unique source as a temporary view
df_unique_source.createOrReplaceTempView("source_flights")

# 3. Build your fully qualified table name
target_table_full = f"{catalog_name}.{silver_schema_name}.lab4_flights_clean"

# 4. Execute the MERGE (Upsert) using pure SQL safely
spark.sql(f"""
    MERGE INTO {target_table_full} AS target
    USING source_flights AS source
    ON target.icao24 = source.icao24
    WHEN MATCHED THEN
      UPDATE SET *
    WHEN NOT MATCHED THEN
      INSERT *
""")

print(f"✅ Delta MERGE executed successfully without conflicts on: {target_table_full}")

# 5. Quick validation preview
display(spark.table(target_table_full).limit(10))

In [0]:
# Ensure unique source records CD Type 1 MERGE
df_cleaned.dropDuplicates(["icao24"]).createOrReplaceTempView("source_flights")
target_table_full = f"{catalog_name}.{silver_schema_name}.lab4_flights_clean"

spark.sql(f"""
    MERGE INTO {target_table_full} AS target
    USING source_flights AS source
    ON target.icao24 = source.icao24
    WHEN MATCHED THEN
      UPDATE SET 
        target.callsign = source.callsign,
        target.longitude = source.longitude,
        target.latitude = source.latitude,
        target.baro_altitude = source.baro_altitude,
        target.velocity = source.velocity,
        target.silver_processed_timestamp = current_timestamp()
    WHEN NOT MATCHED THEN
      INSERT (icao24, callsign, origin_country, longitude, latitude, baro_altitude, velocity, true_track, vertical_rate, on_ground, silver_processed_timestamp, data_layer, source_table)
      VALUES (source.icao24, source.callsign, source.origin_country, source.longitude, source.latitude, source.baro_altitude, source.velocity, source.true_track, source.vertical_rate, source.on_ground, current_timestamp(), source.data_layer, source.source_table)
""")

print("✅ SCD Type 1 MERGE completed.")

In [0]:
# 1. Ensure the schema exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{silver_schema_name}")

# 2. Define target table path
scd2_table_full = f"{catalog_name}.{silver_schema_name}.lab4_flights_scd2"

# 3. Create initial empty SCD Type 2 table with the required tracking columns if it doesn't exist
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {scd2_table_full} (
        icao24 STRING,
        callsign STRING,
        origin_country STRING,
        longitude DOUBLE,
        latitude DOUBLE,
        baro_altitude DOUBLE,
        velocity DOUBLE,
        true_track DOUBLE,
        vertical_rate DOUBLE,
        on_ground BOOLEAN,
        effective_start_date TIMESTAMP,
        effective_end_date TIMESTAMP,
        is_current BOOLEAN
    ) USING DELTA
""")

print(f"✅ SCD Type 2 table ready at: {scd2_table_full}")

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

# 1. Prepare your incoming data by adding the SCD2 tracking columns
# (We use df_cleaned from your earlier cleaning step)
incoming_staged = (
    df_cleaned.dropDuplicates(["icao24"])
    .withColumn("effective_start_date", F.current_timestamp())
    .withColumn("effective_end_date", F.lit(None).cast("timestamp"))
    .withColumn("is_current", F.lit(True))
)

# Register the incoming staged data as a temporary view for SQL merge
incoming_staged.createOrReplaceTempView("source_scd2")

# 2. Reference your target SCD2 table
target_scd2_full = f"{catalog_name}.{silver_schema_name}.lab4_flights_scd2"
delta_table = DeltaTable.forName(spark, target_scd2_full)

# 3. STEP A: EXPIRE existing current records if tracked attributes have changed
# (If icao24 matches an existing active record, but altitude, origin country, or callsign changed)
spark.sql(f"""
    MERGE INTO {target_scd2_full} AS target
    USING source_scd2 AS source
    ON target.icao24 = source.icao24 AND target.is_current = true
    WHEN MATCHED AND (
        target.baro_altitude != source.baro_altitude OR 
        target.origin_country != source.origin_country OR 
        target.callsign != source.callsign
    ) THEN
      UPDATE SET 
        target.is_current = false,
        target.effective_end_date = current_timestamp()
""")

# 4. STEP B: INSERT new records and new versions
# Insert records that either don't exist yet OR were just expired (so a new active row is created)
spark.sql(f"""
    MERGE INTO {target_scd2_full} AS target
    USING source_scd2 AS source
    ON target.icao24 = source.icao24 AND target.is_current = true
    WHEN NOT MATCHED THEN
      INSERT (
          icao24, callsign, origin_country, longitude, latitude, 
          baro_altitude, velocity, true_track, vertical_rate, on_ground, 
          effective_start_date, effective_end_date, is_current
      )
      VALUES (
          source.icao24, source.callsign, source.origin_country, source.longitude, source.latitude, 
          source.baro_altitude, source.velocity, source.true_track, source.vertical_rate, source.on_ground, 
          source.effective_start_date, source.effective_end_date, source.is_current
      )
""")

print(f"✅ SCD Type 2 MERGE completed successfully on: {target_scd2_full}")

# 5. Quick validation preview showing active and historical rows
display(spark.table(target_scd2_full).orderBy("icao24", "effective_start_date").limit(10))

In [0]:
# Schema Evolution and Enhancement.
from pyspark.sql import types as T

# 1. Define a schema that adds an unexpected new column ('pilot_name')
evolved_schema = df_cleaned.schema.add("pilot_name", T.StringType(), True)

# 2. Grab a few rows from df_cleaned and append a dummy value for 'pilot_name'
sample_rows = df_cleaned.limit(5).collect()
malformed_rows = [list(row) + ["Captain Test"] for row in sample_rows]

# 3. Create the malformed DataFrame
df_malformed = spark.createDataFrame(malformed_rows, schema=evolved_schema)

target_table_full = f"{catalog_name}.{silver_schema_name}.lab4_flights_clean"

# --- TEST 1: Schema Enforcement (Should Fail) ---
try:
  df_malformed.write.format("delta").mode("append").saveAsTable(
      target_table_full
  )
  print("⚠️ Write succeeded unexpectedly.")
except Exception as e:
  print("✅ Schema Enforcement successfully blocked the incompatible write!")

# --- TEST 2: Schema Evolution (Should Succeed) ---
(
    df_malformed.write.format("delta")
    .mode("append")
    .option("mergeSchema", "true")  # 👈 Allows the new column to be added
    .saveAsTable(target_table_full)
)

print(
    f"✅ Schema Evolution successful! New column merged into:"
    f" {target_table_full}"
)

# Preview the evolved table
display(spark.table(target_table_full).limit(5))

In [0]:
# Delta column mapping

target_table_full = f"{catalog_name}.{silver_schema_name}.lab4_flights_clean"

# 1. Enable column mapping mode on the table
spark.sql(f"""
    ALTER TABLE {target_table_full} 
    SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name')
""")

# 2. Safely DROP a column (using IF EXISTS to prevent errors if already dropped)
spark.sql(f"""
    ALTER TABLE {target_table_full} DROP COLUMN IF EXISTS pilot_name
""")

# 3. Safely RENAME a column (e.g., renaming 'baro_altitude' to 'altitude')
spark.sql(f"""
    ALTER TABLE {target_table_full} 
    RENAME COLUMN baro_altitude TO altitude
""")

print("✅ Column mapping successful: Dropped and renamed safely without rewriting files!")

# Quick verification preview
display(spark.table(target_table_full).limit(5))

A Data Contract is an explicit agreement between data producers and consumers that defines the schema, types, and quality standards of data before it enters a pipeline. It serves as the controlled alternative to silent schema evolution (mergeSchema = true):

The Problem with Silent Evolution: It automatically accepts upstream changes (like typos, renamed columns, or altered data types), which silently breaks downstream dashboards, ML models, and metrics without warning.

The Data Contract Solution: It shifts validation to the ingestion boundary, rejecting non-compliant data outright rather than mutating table structures. This forces explicit coordination, versioning, and accountability between teams before breaking changes ever hit production.

In [0]:
# ==========================================
# 05. TABLE MAINTENANCE & OPTIMIZATION
# ==========================================

target_table_full = f"{catalog_name}.{silver_schema_name}.lab4_flights_clean"

# 1. Apply Liquid Clustering (Modern alternative to partitioning and Z-ORDER)
# We cluster by the unique flight key and origin country for fast lookups
spark.sql(f"ALTER TABLE {target_table_full} CLUSTER BY (icao24, origin_country)")

# 2. Compact small files using OPTIMIZE
spark.sql(f"OPTIMIZE {target_table_full}")

# 3. Vacuum obsolete file versions (reclaims storage by purging old file versions past retention)
spark.sql(f"VACUUM {target_table_full} RETAIN 168 HOURS")  # 168 hours = 7 days

print(
    f"✅ Step 5 Complete! Liquid Clustering, OPTIMIZE & VACUUM executed"
    f" successfully on: {target_table_full}"
)